# Exploring different multiple regression models for house price prediction

In this notebook we will use data on house sales in King County to predict prices using multiple regression. This first assignment will be about exploring multiple regression in particular exploring the impact of adding features to a regression and measuring error. In the second assignment we will implement a gradient descent algorithm. In this assignment we will:

- Do some feature engineering
- Use built-in functions to compute the regression weights (coefficients)
- Given the regression weights, predictors and outcome write a function to compute the Residual Sum of Squares
- Look at coefficients and interpret their meanings
- Evaluate multiple models via RSS

## Importing libraries

In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## Loading Data

In [4]:
data = pd.read_csv("../../data/kc_house_data.csv")
train_data = pd.read_csv("../../data/kc_house_train_data.csv")
test_data = pd.read_csv("../../data/kc_house_test_data.csv")

## Adding New Variables

Although we often think of multiple regression as including multiple different features (e.g. # of bedrooms, square feet, and # of bathrooms) but we can also consider transformations of existing variables e.g. the log of the square feet or even "interaction" variables such as the product of bedrooms and bathrooms.

Adding 4 new variables in both your train_data and test_data:
- Squaring bedrooms will increase the separation between not many bedrooms (e.g. 1) and lots of bedrooms (e.g. 4) since 1^2 = 1 but 4^2 = 16. Consequently this variable will mostly affect houses with many bedrooms.
- Bedrooms times bathrooms is what's called an "interaction" variable. It is large when both of them are large.
- Taking the log of square feet has the effect of bringing large values closer together and spreading out small values.
- Adding latitude to longitude is non-sensical but we will do it anyway (you'll see why)

In [5]:
for dataset in [train_data, test_data]:
    dataset['bedrooms_squared'] = dataset['bedrooms'] * dataset['bedrooms']
    dataset['bed_bath_rooms'] = dataset['bedrooms'] * dataset['bathrooms']
    dataset['log_sqft_living'] = np.log(dataset['sqft_living'])
    dataset['lat_plus_long'] = dataset['lat'] + dataset['long']

**Quiz Question**: what are the mean (arithmetic average) values of your 4 new variables on TEST data? (round to 2 digits)

In [13]:
new_variables = ['bedrooms_squared', 'bed_bath_rooms', 'log_sqft_living', 'lat_plus_long']
print ( "Answers" )
for variable in new_variables:
    print (variable , "mean\t:" , np.mean(test_data[variable]).round(decimals=2) )

Answers
bedrooms_squared mean	: 12.45
bed_bath_rooms mean	: 7.5
log_sqft_living mean	: 7.55
lat_plus_long mean	: -74.65


## Estimating Coefficients for 3 Models

Use any regression library/function) to estimate the regression coefficients/weights for predicting ‘price’ for the following three models:

- Model 1: ‘sqft_living’, ‘bedrooms’, ‘bathrooms’, ‘lat’, and ‘long’
- Model 2: ‘sqft_living’, ‘bedrooms’, ‘bathrooms’, ‘lat’,‘long’, and ‘bed_bath_rooms’
- Model 3: ‘sqft_living’, ‘bedrooms’, ‘bathrooms’, ‘lat’,‘long’, ‘bed_bath_rooms’, ‘bedrooms_squared’, ‘log_sqft_living’, and ‘lat_plus_long’

You’ll note that the three models here are “nested” in that all of the features of the Model 1 are in Model 2 and all of the features of Model 2 are in Model 3.  

In [43]:
M1 = ['sqft_living', 'bedrooms', 'bathrooms', 'lat', 'long']
M2 = M1 + ['bed_bath_rooms']
M3 = M2 + ['bedrooms_squared', 'log_sqft_living', 'lat_plus_long']

models = {}

for i, cols in enumerate([M1, M2, M3], 1):
    model = LinearRegression().fit(train_data[cols], train_data['price'])
    models[f'M{i}'] = model

**Quiz Question**: What is the sign (positive or negative) for the coefficient/weight for ‘bathrooms’ in Model 1?

In [44]:
print ( "Answer: ", models['M1'].coef_[2] )

Answer:  15706.742082734694


**Quiz Question**: What is the sign (positive or negative) for the coefficient/weight for ‘bathrooms’ in Model 2?

In [45]:
print ( "Answer: ", models['M2'].coef_[2] )

Answer:  -71461.30829275955


## Computing the RSS on Training Data

In [57]:
models_features = { 'M1':M1, 'M2':M2, 'M3':M3 }

for i in range(1, 4):
    key = f'M{i}'

    X_test = train_data[models_features[key]]
    y_test = train_data['price']

    y_pred = models[key].predict(X_test)

    RSS = ((y_test - y_pred) ** 2).sum()
    print ("Model", i, "RSS:", RSS, "\n")

Model 1 RSS: 967879963049545.5 

Model 2 RSS: 958419635074069.1 

Model 3 RSS: 903436455050477.5 



## Computing the RSS on Testing Data

In [56]:
models_features = { 'M1':M1, 'M2':M2, 'M3':M3 }

for i in range(1, 4):
    key = f'M{i}'

    X_test = test_data[models_features[key]]
    y_test = test_data['price']

    y_pred = models[key].predict(X_test)

    RSS = ((y_test - y_pred) ** 2).sum()
    print ("Model", i, "RSS:", RSS, "\n")

Model 1 RSS: 225500469795490.0 

Model 2 RSS: 223377462976467.2 

Model 3 RSS: 259236319207179.66 

